In [1]:
print("Hello, World!")

Hello, World!


In [8]:
import numpy as np 
import pandas as pd
import math
from math import log, sqrt, exp, erf

In [11]:


# =========================================================
# 1) Basic normal CDF
# =========================================================
def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


# =========================================================
# 2) GBM simulation under risk-neutral Black-Scholes
# =========================================================
def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / n_steps

    Z = rng.standard_normal((n_paths, n_steps))
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

    logS = np.cumsum(increments, axis=1)
    logS = np.hstack([np.zeros((n_paths, 1)), logS])

    S = S0 * np.exp(logS)
    return S


# =========================================================
# 3) Vanilla payoffs
# =========================================================
def call_payoff(ST, K):
    return np.maximum(ST - K, 0.0)

def put_payoff(ST, K):
    return np.maximum(K - ST, 0.0)


# =========================================================
# 4) Barrier indicators
# =========================================================
def barrier_indicator(paths, barrier_type, B):
    path_min = np.min(paths, axis=1)
    path_max = np.max(paths, axis=1)

    if barrier_type == "down-and-out":
        return (path_min > B).astype(float)
    elif barrier_type == "down-and-in":
        return (path_min <= B).astype(float)
    elif barrier_type == "up-and-out":
        return (path_max < B).astype(float)
    elif barrier_type == "up-and-in":
        return (path_max >= B).astype(float)
    else:
        raise ValueError(f"Unknown barrier_type: {barrier_type}")


# =========================================================
# 5) Generic Monte Carlo barrier pricer
# =========================================================
def mc_barrier_option(paths, K, r, T, option_kind, barrier_type, B):
    ST = paths[:, -1]

    if option_kind == "call":
        vanilla = call_payoff(ST, K)
    elif option_kind == "put":
        vanilla = put_payoff(ST, K)
    else:
        raise ValueError("option_kind must be 'call' or 'put'")

    ind = barrier_indicator(paths, barrier_type, B)
    payoff = vanilla * ind

    disc_payoff = np.exp(-r * T) * payoff
    price = np.mean(disc_payoff)
    std_error = np.std(disc_payoff, ddof=1) / np.sqrt(len(disc_payoff))
    return price, std_error


# =========================================================
# 6) Named wrappers for the 8 barrier options
# =========================================================
def mc_down_and_out_call(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "call", "down-and-out", B)

def mc_down_and_in_call(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "call", "down-and-in", B)

def mc_up_and_out_call(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "call", "up-and-out", B)

def mc_up_and_in_call(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "call", "up-and-in", B)

def mc_down_and_out_put(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "put", "down-and-out", B)

def mc_down_and_in_put(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "put", "down-and-in", B)

def mc_up_and_out_put(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "put", "up-and-out", B)

def mc_up_and_in_put(paths, K, B, r, T):
    return mc_barrier_option(paths, K, r, T, "put", "up-and-in", B)


# =========================================================
# 7) Monte Carlo floating-strike lookback options
#    Proposition 9.1 and 9.5 correspond to:
#    Put payoff  = M_T - S_T
#    Call payoff = S_T - m_T
# =========================================================
def mc_lookback_put(paths, r, T):
    ST = paths[:, -1]
    path_max = np.max(paths, axis=1)
    payoff = path_max - ST

    disc_payoff = np.exp(-r * T) * payoff
    price = np.mean(disc_payoff)
    std_error = np.std(disc_payoff, ddof=1) / np.sqrt(len(disc_payoff))
    return price, std_error

def mc_lookback_call(paths, r, T):
    ST = paths[:, -1]
    path_min = np.min(paths, axis=1)
    payoff = ST - path_min

    disc_payoff = np.exp(-r * T) * payoff
    price = np.mean(disc_payoff)
    std_error = np.std(disc_payoff, ddof=1) / np.sqrt(len(disc_payoff))
    return price, std_error


# =========================================================
# 8) Closed-form vanilla Black-Scholes
# =========================================================
def bs_call_price(S0, K, r, sigma, T):
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S0 * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)

def bs_put_price(S0, K, r, sigma, T):
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S0 * norm_cdf(-d1)


# =========================================================
# 9) Closed-form barrier prices (continuous monitoring)
#    Standard Reiner-Rubinstein-style formulas with q = 0
#
#    phi = +1 for call, -1 for put
#    eta = +1 for down, -1 for up
# =========================================================
def barrier_closed_form(S0, K, B, r, sigma, T, option_kind, barrier_type):
    if not (B > 0 and S0 > 0 and K > 0 and sigma > 0 and T > 0):
        raise ValueError("Inputs must satisfy B,S0,K,sigma,T > 0")

    phi = 1 if option_kind == "call" else -1
    eta = 1 if barrier_type.startswith("down") else -1

    mu = (r - 0.5 * sigma**2) / sigma**2
    sigT = sigma * math.sqrt(T)

    # Standard building blocks
    x1 = math.log(S0 / K) / sigT + (1.0 + mu) * sigT
    x2 = math.log(S0 / B) / sigT + (1.0 + mu) * sigT
    y1 = math.log((B * B) / (S0 * K)) / sigT + (1.0 + mu) * sigT
    y2 = math.log(B / S0) / sigT + (1.0 + mu) * sigT

    A = phi * S0 * norm_cdf(phi * x1) - phi * K * math.exp(-r * T) * norm_cdf(phi * x1 - phi * sigT)
    B_term = phi * S0 * norm_cdf(phi * x2) - phi * K * math.exp(-r * T) * norm_cdf(phi * x2 - phi * sigT)
    C = (
        phi * S0 * (B / S0) ** (2.0 * (mu + 1.0)) * norm_cdf(eta * y1)
        - phi * K * math.exp(-r * T) * (B / S0) ** (2.0 * mu) * norm_cdf(eta * y1 - eta * sigT)
    )
    D = (
        phi * S0 * (B / S0) ** (2.0 * (mu + 1.0)) * norm_cdf(eta * y2)
        - phi * K * math.exp(-r * T) * (B / S0) ** (2.0 * mu) * norm_cdf(eta * y2 - eta * sigT)
    )

    # Case-by-case mapping consistent with Table 8.1
    if option_kind == "call" and barrier_type == "down-and-out":
        return A - C if B <= K else B_term - D

    if option_kind == "call" and barrier_type == "down-and-in":
        return C if B <= K else A - B_term + D

    if option_kind == "call" and barrier_type == "up-and-out":
        return 0.0 if B <= K else A - B_term + C - D

    if option_kind == "call" and barrier_type == "up-and-in":
        return A if B <= K else B_term - C + D

    if option_kind == "put" and barrier_type == "down-and-out":
        return A - B_term + C - D if B <= K else 0.0

    if option_kind == "put" and barrier_type == "down-and-in":
        return B_term - C + D if B <= K else A

    if option_kind == "put" and barrier_type == "up-and-out":
        return B_term - D if B <= K else A - C

    if option_kind == "put" and barrier_type == "up-and-in":
        return A - B_term + D if B <= K else C

    raise ValueError("Invalid combination of option_kind and barrier_type")


# =========================================================
# 10) Closed-form floating-strike lookback prices
#     Fresh contract at t = 0, so running min/max start at S0
#
#     Prop 9.1: payoff M_T - S_T
#     Prop 9.5: payoff S_T - m_T
#
#     These formulas assume r != 0
# =========================================================
def lookback_put_closed_t0(S0, r, sigma, T):
    """
    Floating-strike lookback put:
        payoff = M_T - S_T
    """
    if abs(r) < 1e-12:
        raise ValueError("This implementation assumes r != 0.")

    d1 = (r + 0.5 * sigma**2) * T / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    lam = sigma**2 / (2.0 * r)

    price = S0 * (
        (1.0 + lam) * norm_cdf(d1)
        + math.exp(-r * T) * (1.0 - lam) * norm_cdf(-d2)
        - 1.0
    )
    return price


def lookback_call_closed_t0(S0, r, sigma, T):
    """
    Floating-strike lookback call:
        payoff = S_T - m_T
    """
    if abs(r) < 1e-12:
        raise ValueError("This implementation assumes r != 0.")

    d1 = (r + 0.5 * sigma**2) * T / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    lam = sigma**2 / (2.0 * r)

    price = S0 * (
        norm_cdf(d1)
        - math.exp(-r * T) * norm_cdf(d2)
        + math.exp(-r * T) * lam * norm_cdf(d2)
        - lam * norm_cdf(-d1)
    )
    return price


# =========================================================
# 11) Optional MC vanilla prices for parity checks
# =========================================================
def mc_vanilla_option(paths, K, r, T, option_kind):
    ST = paths[:, -1]
    if option_kind == "call":
        payoff = call_payoff(ST, K)
    elif option_kind == "put":
        payoff = put_payoff(ST, K)
    else:
        raise ValueError("option_kind must be 'call' or 'put'")

    disc_payoff = np.exp(-r * T) * payoff
    return np.mean(disc_payoff), np.std(disc_payoff, ddof=1) / np.sqrt(len(disc_payoff))




In [10]:

# Main experiment
if __name__ == "__main__":
    # ---------------------------
    # Parameters
    # ---------------------------
    S0 = 100.0
    K = 100.0
    r = 0.05
    sigma = 0.20
    T = 1.0

    # Down barrier for down options, up barrier for up options
    B_down = 90.0
    B_up = 110.0

    n_paths = 200_000
    n_steps = 252
    seed = 42

    # ---------------------------
    # Simulate paths
    # ---------------------------
    paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, seed=seed)

    # ---------------------------
    # Barrier comparisons
    # ---------------------------
    barrier_specs = [
        ("Down-and-out Call", "call", "down-and-out", B_down),
        ("Down-and-in Call",  "call", "down-and-in",  B_down),
        ("Up-and-out Call",   "call", "up-and-out",   B_up),
        ("Up-and-in Call",    "call", "up-and-in",    B_up),
        ("Down-and-out Put",  "put",  "down-and-out", B_down),
        ("Down-and-in Put",   "put",  "down-and-in",  B_down),
        ("Up-and-out Put",    "put",  "up-and-out",   B_up),
        ("Up-and-in Put",     "put",  "up-and-in",    B_up),
    ]

    barrier_rows = []
    for name, opt_kind, bar_type, B in barrier_specs:
        mc_price, mc_se = mc_barrier_option(paths, K, r, T, opt_kind, bar_type, B)
        cf_price = barrier_closed_form(S0, K, B, r, sigma, T, opt_kind, bar_type)
        abs_err = abs(mc_price - cf_price)
        rel_err = abs_err / abs(cf_price) if abs(cf_price) > 1e-12 else np.nan

        barrier_rows.append([
            name, B, mc_price, mc_se,
            mc_price - 1.96 * mc_se,
            mc_price + 1.96 * mc_se,
            cf_price, abs_err, rel_err
        ])

    barrier_df = pd.DataFrame(
        barrier_rows,
        columns=[
            "Option", "Barrier", "MC Price", "Std. Error",
            "95% CI Lower", "95% CI Upper",
            "Closed Form", "Abs Error", "Rel Error"
        ]
    )

    # ---------------------------
    # Lookback comparisons
    # ---------------------------
    lb_put_mc, lb_put_se = mc_lookback_put(paths, r, T)
    lb_call_mc, lb_call_se = mc_lookback_call(paths, r, T)

    lb_put_cf = lookback_put_closed_t0(S0, r, sigma, T)
    lb_call_cf = lookback_call_closed_t0(S0, r, sigma, T)

    lookback_rows = [
        [
            "Floating Lookback Put  (M_T - S_T)",
            lb_put_mc, lb_put_se,
            lb_put_mc - 1.96 * lb_put_se,
            lb_put_mc + 1.96 * lb_put_se,
            lb_put_cf,
            abs(lb_put_mc - lb_put_cf),
            abs(lb_put_mc - lb_put_cf) / abs(lb_put_cf)
        ],
        [
            "Floating Lookback Call (S_T - m_T)",
            lb_call_mc, lb_call_se,
            lb_call_mc - 1.96 * lb_call_se,
            lb_call_mc + 1.96 * lb_call_se,
            lb_call_cf,
            abs(lb_call_mc - lb_call_cf),
            abs(lb_call_mc - lb_call_cf) / abs(lb_call_cf)
        ],
    ]

    lookback_df = pd.DataFrame(
        lookback_rows,
        columns=[
            "Option", "MC Price", "Std. Error",
            "95% CI Lower", "95% CI Upper",
            "Closed Form", "Abs Error", "Rel Error"
        ]
    )

    # ---------------------------
    # Optional parity checks
    # ---------------------------
    vanilla_call_mc, _ = mc_vanilla_option(paths, K, r, T, "call")
    vanilla_put_mc, _ = mc_vanilla_option(paths, K, r, T, "put")

    parity_df = pd.DataFrame([
        [
            "Down Call In-Out Parity",
            vanilla_call_mc,
            barrier_df.loc[barrier_df["Option"] == "Down-and-out Call", "MC Price"].iloc[0]
            + barrier_df.loc[barrier_df["Option"] == "Down-and-in Call", "MC Price"].iloc[0],
        ],
        [
            "Up Put In-Out Parity",
            vanilla_put_mc,
            barrier_df.loc[barrier_df["Option"] == "Up-and-out Put", "MC Price"].iloc[0]
            + barrier_df.loc[barrier_df["Option"] == "Up-and-in Put", "MC Price"].iloc[0],
        ],
    ], columns=["Check", "Vanilla MC", "In + Out"])

    parity_df["Difference"] = parity_df["Vanilla MC"] - parity_df["In + Out"]

    # ---------------------------
    # Print nicely
    # ---------------------------
    pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

    print("\nPARAMETERS")
    print("-" * 80)
    print(f"S0      = {S0}")
    print(f"K       = {K}")
    print(f"r       = {r}")
    print(f"sigma   = {sigma}")
    print(f"T       = {T}")
    print(f"B_down  = {B_down}")
    print(f"B_up    = {B_up}")
    print(f"n_paths = {n_paths}")
    print(f"n_steps = {n_steps}")

    print("\n8 BARRIER OPTIONS: MONTE CARLO VS CLOSED FORM")
    print("-" * 80)
    print(barrier_df.to_string(index=False))

    print("\nLOOKBACK OPTIONS: MONTE CARLO VS CLOSED FORM")
    print("-" * 80)
    print(lookback_df.to_string(index=False))

    print("\nOPTIONAL PARITY CHECKS")
    print("-" * 80)
    print(parity_df.to_string(index=False))


PARAMETERS
--------------------------------------------------------------------------------
S0      = 100.0
K       = 100.0
r       = 0.05
sigma   = 0.2
T       = 1.0
B_down  = 90.0
B_up    = 110.0
n_paths = 200000
n_steps = 252

8 BARRIER OPTIONS: MONTE CARLO VS CLOSED FORM
--------------------------------------------------------------------------------
           Option    Barrier  MC Price  Std. Error  95% CI Lower  95% CI Upper  Closed Form  Abs Error  Rel Error
Down-and-out Call  90.000000  8.988422    0.033012      8.923719      9.053125     8.665472   0.322950   0.037269
 Down-and-in Call  90.000000  1.516281    0.011951      1.492858      1.539704     1.785112   0.268831   0.150596
  Up-and-out Call 110.000000  0.152332    0.001932      0.148545      0.156119     0.118614   0.033718   0.284267
   Up-and-in Call 110.000000 10.352371    0.033292     10.287119     10.417623    10.331970   0.020401   0.001975
 Down-and-out Put  90.000000  0.190647    0.002178      0.186377      0.